# CIS I — Assignment 2: Kinematic Calibration of a Cartesian Robot

This notebook walks through the calibration scenario from the handout
(Figure 1): a cartesian-base + spherical-wrist robot carrying a special
calibration marker body, observed by an optical tracker. Your job is to
**design** the scene (where the tracker and robot go, what the calibration
body's marker layout is), **collect** data under a simulated clock, and
**fit** an improved kinematic model `ImprovedFwdKins` / `ImprovedInvKins`
that beats the nominal one over the robot's work volume: the handout asks
for `||alpha|| <= 1e-3 rad` and `||epsilon|| <= 0.2 mm`, in as little
simulated time as possible — and if you cannot get there, to report the
accuracy you achieved and how you went about it.

Everything runs against `src.simulation`, a self-contained simulator playing
the role of the physical robot + tracker (the tracker and marker-body
classes are assignment 1's, imported from `assignments/assignment1`). It
injects the kinematic errors and measurement noise — your code
never sees ground truth except where explicitly noted for
**validation/reporting** in Steps 8–10.

Unlike assignment 1, your code lives in Python modules under `src/student/`
(every function there is a stub raising `NotImplementedError`), driven from
this notebook and unit-tested from `tests/`. Keep the signatures the
notebook relies on; If you want to design the functions in a different way - feel free to do that. 
But indicate that in your report.
otherwise the design is yours. Remember the assignment's
closing note: this is a design exercise — part of the job is deciding which
items of the tracker's and robot's "spec sheet" matter for your error
budget, and your report must justify those decisions.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation, vct3, Rot, Frame
from src.simulation import clock, constants
from src.student import metrics, registration, workflow, kinematics

np.set_printoptions(precision=4, suppress=True)
clock.reset()

## Step 1: Design the scene

Specify approximate poses (as `Frame`) for the tracking camera and the robot
base, relative to the workspace, and the calibration body's marker layout
(`List[vct3]`, local coordinates). The simulator perturbs the robot's
placement the way the handout says (`||alpha_rob|| <= 0.05`,
`||epsilon_rob|| <= 5 mm`), so `F_rob_approx` is a design value, not truth.

Things to keep in mind:

- The tracker's usable volume (Figure 2) is a frustum along its **local
  +z axis**: 100–220 cm in front of the camera, widening from a 100 cm to a
  200 cm square. `trk.is_visible(points)` checks it for you.
- This handout's tracker has **no jiggle** (`sigma_Ta = sigma_Tp = 0`): the
  pose you give it is exactly where it is, so the tracker frame is a fixed
  reference and everything you measure is expressed in it. What remains is
  0.1 mm of random noise per marker reading and a small systematic field
  (`<= 0.1 mm`, slowly varying).
- The robot's end effector sits 600 mm along the base x-axis at home, and
  the linear joints travel `0..600, -300..300, 0..600` mm — so the work
  volume is a 600 mm cube spanning x ∈ [600, 1200], y ∈ [−300, 300],
  z ∈ [0, 600] in base coordinates, and the wrist (±0.75π, ±0.75π, ±π)
  swings the 75 mm calibration body around inside it. Everything must stay
  visible.
- The calibration body's layout is exact (`a*_k == a_k`) but limited to
  `||a*_k|| <= 75 mm`. Its orientation is estimated from its markers, so
  the number of markers and how far they spread set how noisy each
  orientation reading is.
- You may define and place additional marker bodies
  (`simulation.define_marker_body` / `place_marker_body`) if your design
  wants them; their layouts are only known to ±2 mm (see Step 5).

In [2]:
# Robot base at the workspace origin. The base-pose uncertainty moves a point p by up to
# eps_max_rob + alpha_max_rob |p| (5 mm + 0.05 |p|), so keeping the work volume near the origin
# keeps the visibility margin small.
F_rob_approx = Frame.eye()

# Tracker: side view of the work volume from +y, about 1.7 m from its center (840, 0, 200) mm.
# Chosen by a grid search over viewing directions and distances: every calibration stop keeps
# more than 200 mm of clearance from the viewing-volume faces beyond that worst-case base shift.
# The camera looks along its local +z.
F_tracker_approx = Frame(Rot(matrix=[[0.9397, 0.0, 0.3420],
                                     [0.3420, 0.0, -0.9397],
                                     [0.0, 1.0, 0.0]]).normalize(),
                         vct3(258.6, 1597.5, 200.0))

# Calibration body: a regular tetrahedron of radius 74.9 mm (just inside the 75 mm limit) centred
# on the origin of {e}. The isotropic layout (M = 2/3 N r^2 I) makes the orientation noise equal
# about every axis and lets no symmetric part of the tracker-distortion gradient leak into the
# rotation; the {e} origin at the marker centroid minimises the position noise (report Section 3).
r_mbee = 74.9
mbee_local_positions = [vct3(r_mbee * np.array(v) / np.sqrt(3))
                        for v in ([1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1])]

## Step 2: Build the simulation

Fully provided — this wires your Step 1 choices through the simulator's
`CreateRobot` / `CreateTracker` / `CreateRobotCalibrationBody` /
`AddMarkerBodies` equivalents. The robot's hidden kinematic errors and the
tracker's hidden systematic-error field are drawn from `SEED`; your
calibration must work for *any* seed (grading uses different ones), so
don't tune to it.

`mbee_placed` is a `PlacedBody` handle pairing the calibration body with its
hidden actual placement; the body starts attached and moves with the robot.
Note that `rob.state()` already reports non-zero encoder values: the robot
starts at its physical home, `theta* = 0`, and the encoders read
`-delta_theta_sys(home) - delta_theta_rnd` there (see `rob.home_robot()`).

In [3]:
SEED = 1

rob = simulation.create_robot("rob", F_rob_approx, seed=SEED)
trk = simulation.create_tracker("trk", F_tracker_approx, seed=SEED)
mbee_placed = simulation.create_robot_calibration_body(rob, mbee_local_positions)
simulation.add_marker_bodies(trk, [mbee_placed])

print("simulation built:", rob, "|", trk.name, "| tracking", [p.body.name for p in trk.tracked_bodies])
print("robot state:", rob.state())

simulation built: Robot(name='rob', theta_cmd=[-0.0323, -0.0808, 0.0449, -0.0073, -0.0086, 0.0078]) | trk | tracking ['mBee']
robot state: RobotState(theta=array([-0.0323, -0.0808,  0.0449, -0.0073, -0.0086,  0.0078]), theta_dot=array([0., 0., 0., 0., 0., 0.]), t_elapsed=0.0, calibration_body_attached=True)


### Step 2.1: Visibility check

Before spending any simulated time, sweep a joint grid and check that
the calibration body stays inside the tracker's volume (using the *nominal*
placements — they're only tens of mm off, which is nothing against the
frustum size). If a large fraction is invisible, go back to Step 1.

In [4]:
def mbee_markers_world(theta):
    F_ee_world = rob.F_rob * rob.fwd_kins(theta)
    return [F_ee_world * a for a in mbee_local_positions]

n_vis, n_tot = 0, 0
for x in (0, 300, 600):
    for y in (-300, 0, 300):
        for z in (0, 300, 600):
            for w in (-1.8, 0.0, 1.8):
                theta = np.array([x, y, z, w, -w, w])
                n_tot += 1
                n_vis += trk.is_visible(mbee_markers_world(theta))
print(f"calibration body visible at {n_vis}/{n_tot} corner/edge poses of the work volume")

calibration body visible at 81/81 corner/edge poses of the work volume


## Step 3: Unit tests

`tests/test_simulation.py` is provided and passes as shipped — read it as
an example. `tests/test_student.py` exercises *your* `src/student` code:
the complete tests there fail with `NotImplementedError` until you
implement what they call, and the ones marked `TODO` are yours to write (you are required to write unit tests of each significant element, described in your
report together with their results). Re-run this cell as you go.

In [5]:
!{sys.executable} -m pytest ../tests -q -p no:cacheprovider

..............................................                           [100%]
46 passed in 9.90s


## Step 4: Error metric

You have to measure the mismatch between two frames as
`dF = F_a^-1 F_b ≈ Frame(I + sk(alpha), epsilon)`. Implement
`metrics.frame_error` (returning `||alpha||`, `||epsilon||`) and
`metrics.summarize_errors`; every accuracy number below goes through them.

In [6]:
print("sanity:", metrics.frame_error(Frame.eye(), Frame(Rot('z', 0.01), vct3(1, 0, 0))))

sanity: (0.01, 1.0)


## Step 5 (optional): Registration and marker-body refinement

> "It is possible to improve your estimate of a_m,k through a calibration
> process involving multiple observations of the marker body using the
> tracking camera and then averaging them."

The calibration body needs no refinement — `a*_k == a_k` by construction —
so this step only matters if your design places *additional* marker bodies
(their layouts are only known to ±2 mm, and the tracker registers each
body against its *current* nominal layout). You will need your own
`registration.register_points` (Arun's method — do not call the
simulator's solver) and `registration.average_frames` for Step 6 in any
case; `registration.refine_marker_body` is the optional part.

In [7]:
print("register_points sanity:", registration.register_points(
    [vct3(0, 0, 0), vct3(50, 0, 0), vct3(0, 50, 0), vct3(0, 0, 50)],
    [vct3(10, 0, 0), vct3(60, 0, 0), vct3(10, 50, 0), vct3(10, 0, 50)]).p)

register_points sanity: vct3(x=10.00, y=0.00, z=-0.00)
Internal Vector:
[[10.]
 [ 0.]
 [-0.]]


## Step 6: Home the robot, then a baseline in the tracker frame

> "`theta_encoders = HomeRobot()` will place the robot at a home position in
> which `theta*_k = 0` and return the current values read by the encoders.
> This may prove useful as part of your calibration process."

The encoders read `theta = theta* - delta_theta_rnd - delta_theta_sys`, so at
the physical home they expose the systematic offset of every joint
directly. Implement `workflow.measure_home_offsets` (average a few homings).

Then the baseline: the calibration body's coordinate system is exactly
`F_ee`, so every reading satisfies

    F_ee,j  =  F_X · FwdKins(theta_j)

for one constant unknown `F_X` (the robot base expressed in the tracker's
frame). Implement `workflow.collect_calibration_data` (robot move + one
`sample_marker_bodies` reading, storing the body's frame) and
`workflow.estimate_base_frame` (you'll want `registration.average_frames`),
then look at how well the **nominal** kinematics explain even a handful of
poses.

In [8]:
t0 = clock.elapsed
home_offsets = workflow.measure_home_offsets(rob, n_homings=5) # you can change the number of homings
t_home = clock.elapsed - t0
print("delta_theta_sys at home (from the encoders):", home_offsets)

center = np.array([300.0, 0.0, 300.0, 0.0, 0.0, 0.0])
baseline_poses = [center] + [center + np.eye(6)[k] * v for k in range(3) for v in (-250.0, 250.0)] \
                 + [center + np.eye(6)[3] * 0.8, center + np.eye(6)[4] * 0.8, center + np.eye(6)[5] * 1.5]

baseline = workflow.collect_calibration_data(rob, trk, mbee_placed, baseline_poses, repeats=1)
t_baseline = baseline.t_collect
F_X_baseline = workflow.estimate_base_frame(baseline, rob)
residuals = [metrics.frame_error(F_obs, F_X_baseline * rob.fwd_kins(th)) for th, F_obs in zip(baseline.thetas, baseline.frames)]
print(f"{len(baseline)} baseline samples in {baseline.t_collect:.1f}s")
print("nominal kinematics vs observations:", metrics.summarize_errors(residuals))

delta_theta_sys at home (from the encoders): [ 0.0299  0.0803 -0.0474  0.0074  0.0086 -0.0079]
10 baseline samples in 83.8s
nominal kinematics vs observations: {'n': 10, 'rms_angle': 0.008117751855358053, 'mean_angle': 0.006440803730897076, 'max_angle': 0.01913355523760939, 'rms_pos': 2.8855381506175535, 'mean_pos': 2.401647971851292, 'max_pos': 4.983004228271714}


## Step 7: Plan and collect the calibration data

> "The goal is to reach a specified accuracy within as short a time as
> possible." — `clock.elapsed` is the score's time component.

The handout suggests three routes and *strongly* urges the first:

1. **Estimate the kinematic parameters.** (a) Work out how closely spaced
   your samples along an axis must be for the accuracy you want — this
   depends on the error bounds `delta_sys^trk`, `omega_sys^trk`,
   `delta_sys^rob`, `omega_sys^rob`; (b) home the robot, then systematically
   move along/about each individual axis while taking data, returning home
   after each axis; (c) determine the angles between the axes and the
   `theta -> theta*` map (use any interpolation scheme you find feasible); (d) write the corrected
   forward and inverse kinematics; (e) report the updated parameters and,
   in debugging, compare them with ground truth; then "bottom line" test a
   sampled portion of the workspace.
2. **Map the space and interpolate** — many more, closely spaced points,
   and dense random-point testing.
3. A mix.

Design `workflow.plan_calibration_poses`: which joint vectors to visit, in
which order. Every parameter of your model must be excited (axis tilts need
travel along each axis; wrist axes, `F_we` and the wrist joints' distortion
need wrist rotations; the linear joints' distortion needs stops along each
joint), every pose must keep the calibration body visible, and the wrist
moves at only 0.1 rad/s. Joints 1–3 are kinematically decoupled from 4–6.

`plan_calibration_poses` takes the accuracy you are aiming for and must
size the plan from it: how many distinct poses, and how many readings of
each (a pose listed twice in a row is re-commanded, costing only `t_settle`
— which also re-rolls the joint repeatability noise — and re-read for
`delta_t_samp`). With 0.1 mm of marker noise on a <= 75 mm body, how many
readings does a 1e-3 rad orientation estimate need? Work it out before you
spend simulated time.

In [9]:
DESIRED_ACCURACY = constants.TARGET_ANG_ERR_RAD  # rad; the orientation target is the limiting one

# Handout method 1 (b), one joint at a time: each joint is swept through its knots (100 mm / pi/4
# apart, report Section 3) starting from, and returning to, the encoder configuration theta = 0
# after each axis. The sweeps do not start at the HomeRobot() reading itself: that reading,
# -delta_theta_sys - delta_theta_rnd, can be negative on joints 1 and 3, outside their 0..600 mm
# limits, so MoveJoints could not sweep the other joints from it; theta = 0 differs from it by at
# most delta_sys (0.2 mm / 0.01 rad), and the home reading still enters the model as the fixed
# home knot. The single-axis sweeps are what the decoupled initial estimate in
# ImprovedKinematics.fit needs. method="concurrent" is the faster variant compared at the end of
# this notebook.
PLAN = dict(method="single_axis")

n_stage, n_wrist = workflow.readings_per_stop(mbee_local_positions, DESIRED_ACCURACY)
print(f"readings per stop: {n_stage} (stages), {n_wrist} (wrist)")
poses = workflow.plan_calibration_poses(rob, trk, mbee_local_positions, desired_accuracy=DESIRED_ACCURACY, **PLAN)
n_distinct = len({tuple(np.round(th, 6)) for th in poses})
print(f"plan: {len(poses)} readings over {n_distinct} distinct poses, "
      f"estimated {workflow.estimate_plan_time(rob, poses):.1f}s")

t0 = clock.elapsed
data = workflow.collect_calibration_data(rob, trk, mbee_placed, poses, repeats=1)
print(f"{len(data)} samples collected in {data.t_collect:.1f}s; clock now {clock.elapsed:.1f}s")

F_X_nom = workflow.estimate_base_frame(data, rob)

readings per stop: 7 (stages), 68 (wrist)
plan: 1716 readings over 39 distinct poses, estimated 543.3s
1716 samples collected in 543.3s; clock now 627.5s


## Step 8: Fit the improved kinematics

This is the assignment's core deliverable — implement
`kinematics.ImprovedKinematics` yourself and describe it in the report.

The handout's error model: `theta*_k = theta_k + delta_theta_k,rnd +
delta_theta_k,sys` on every joint (smooth, `||delta_theta_sys|| <=
delta_sys`, slope `<= omega_sys`), `d*_k = R(phi_k) d_k` for `k = 2..6`
(`d*_1 = d_1`), `F*_we = F_we dF_ee`, plus the base frame `F_X`. Choose a
parameter vector (the class docstring sketches one), write
`fwd_kins(theta, params)`, a residual against `data`, and a non-linear
least-squares `fit` (`scipy.optimize` is fine; so is a numerical
Jacobian). `inv_kins` is provided: it iterates on *your* `fwd_kins`.

**Questions for the report** (your `jacobian_rank_report` should help):
is every parameter identifiable from your poses? Is every parameter in the
handout's model identifiable *at all* — what happens to a component of
`phi_k` along `d_k`, to a rotation of the base about `d_1`, or to a joint
offset on joints 1–3 versus a shift of the base? (Which is why the home
offsets come from `HomeRobot()` rather than from the fit.) Which of the
handout's error terms are big enough to matter at the 0.2 mm / 1e-3 rad
level, and which are below the noise? How did that shape your
parameterization and your pose plan?

**Ground truth for debugging only**: `rob.ground_truth_parameters()`
returns the hidden kinematic parameters so you can compare your
`parameters_report()` with them, as the handout asks for debugging runs —
never use it in the calibration itself.

In [10]:
improved = kinematics.ImprovedKinematics(rob, F_X_nom, joint_offsets=home_offsets)
# decoupled per-axis estimate, weighted non-linear least squares, then the rotation about d_1 that
# no data can determine, chosen from the handout's bounds (F_tracker is the pose specified in Step 1)
fit_info = improved.fit(data, F_tracker=F_tracker_approx)
print("fit:", {k: (float(f"{v:.4g}") if isinstance(v, float) else v) for k, v in fit_info.items()})

rank = improved.jacobian_rank_report(data)
print("jacobian rank:", rank["rank"], "/", rank["n_params"], f"| condition {rank['condition']:.3g}")
print("weakest directions:", [(f"{w['singular_value']:.3g}", w["params"][0][0]) for w in rank["weakest"]])

t_calibration = clock.elapsed
print(f"total simulated calibration time: {t_calibration:.1f}s (soft budget {constants.TARGET_TIME_SEC:.0f}s), "
      f"of which baseline {t_baseline:.1f}s")

estimated, truth = improved.parameters_report(), rob.ground_truth_parameters()   # truth: debugging only
truth_angles = kinematics.ImprovedKinematics.axis_angles_deg(truth["axes"])
print("estimated axis angles (deg):\n", estimated["axis_angles_deg"])
print("axis-angle error (deg), max:", np.abs(estimated["axis_angles_deg"] - truth_angles).max())
print("estimated joint offsets:", estimated["joint_offset"], "| true:", truth["joint_offset"])
print("estimated joint scales :", estimated["joint_scale"], "| true:", truth["joint_scale"])
print("estimated F_we twist   :", estimated["F_we_twist"], "| true:", truth["F_we_twist"])
# The tilts, the x-component of the tool rotation (F_we twist[0]) and the base rotation about d_1
# are known only within fit_info["gauge_interval"] (the rotation about d_1 is not observable);
# compare them through the gauge-free axis angles above. The tracker-frame axis directions are
# gauge-free as well:
print("rotation about d_1: chosen c = %.5f rad, interval %s" % (fit_info["gauge_c"], fit_info["gauge_interval"]))
F_trk = trk.nominal_pose()
R_true = (F_trk.inv() * truth["F_rob_actual"]).R.matrix
R_est = improved.base_frame().R.matrix
dirs_err = [np.degrees(np.arccos(np.clip((R_est @ a) @ (R_true @ b), -1, 1)))
            for a, b in zip(estimated["axes"][:3], truth["axes"][:3])]
print("stage axis directions in the tracker frame, error (deg):", np.round(dirs_err, 5))

fit: {'init': 'decoupled', 'init_rms_rot_residual': 0.000383, 'init_rms_pos_residual': 0.02056, 'cost': 85.35, 'nfev': 9, 'status': 2, 'success': True, 'n_groups': 39, 'rms_rot_residual': 0.0003604, 'rms_pos_residual': 0.01643, 'gauge_c': -0.01519, 'gauge_interval': (-0.017919999999974873, -0.0124599999999732)}
jacobian rank: 59 / 59 | condition 9.11e+03
weakest directions: [('51.3', 'knot2@100.000'), ('51.5', 'knot3@200.000'), ('51.8', 'knot1@100.000')]
total simulated calibration time: 627.5s (soft budget 900s), of which baseline 83.8s
estimated axis angles (deg):
 [[ 0.     90.0648 90.1768  0.7222 91.1351 90.4227]
 [90.0648  0.     88.3094 90.7363  1.3796 88.5215]
 [90.1768 88.3094  0.     89.931  89.1767  0.3249]
 [ 0.7222 90.7363 89.931   0.     91.8107 90.1744]
 [91.1351  1.3796 89.1767 91.8107  0.     89.3841]
 [90.4227 88.5215  0.3249 90.1744 89.3841  0.    ]]
axis-angle error (deg), max: 0.005602672567903255
estimated joint offsets: [ 0.021   0.0471 -0.0645  0.0074  0.0086 -0.

## Step 9: Evaluate `ImprovedFwdKins` over the work volume

**Ground truth for reporting only**: `rob.actual_ee_pose(theta)` (the
actual end-effector pose in the workspace for an encoder command `theta`)
is a hidden value you have access to here only because this notebook
doesn't gate it. Use it *only* in Steps 9–10 to compute and report your
achieved error. With no tracker jiggle, the tracker's true pose is the one
you specified, so `F_trk = trk.nominal_pose()` maps between the workspace
and the tracker frame.

The metric, per test pose `theta_i`, with `F̂_X` your estimated base frame:

    dF_i = (F_trk^-1 · F*_ee(theta_i))^-1 · (F̂_X · ImprovedFwdKins(theta_i))

Report `||alpha||` and `||epsilon||` statistics over the grid below (which
is fixed — don't change it) for the nominal kinematics and for yours, and
compare against `constants.TARGET_POS_ERR_MM` / `TARGET_ANG_ERR_RAD`.
Evaluation reads the oracle only; it must not move the robot or sample the
tracker.

In [11]:
test_levels = {0: (100.0, 300.0, 500.0), 1: (-200.0, 0.0, 200.0), 2: (100.0, 300.0, 500.0),
               3: (-1.3, 0.0, 1.3), 4: (-1.3, 0.0, 1.3), 5: (-2.0, 0.0, 2.0)}
test_grid = [np.array([x, y, z, a, b, c])
             for x in test_levels[0] for y in test_levels[1] for z in test_levels[2]
             for a in test_levels[3] for b in test_levels[4] for c in test_levels[5]]
test_grid = [th for th in test_grid if trk.is_visible(mbee_markers_world(th))]
print(f"{len(test_grid)} visible test poses")

F_trk = trk.nominal_pose()

fwd_err_nominal, fwd_err_improved = [], []
for theta in test_grid:
    truth_pose = F_trk.inv() * rob.actual_ee_pose(theta)          # ground truth, reporting only
    fwd_err_nominal.append(metrics.frame_error(truth_pose, F_X_nom * rob.fwd_kins(theta)))
    fwd_err_improved.append(metrics.frame_error(truth_pose, improved.predict_in_tracker(theta)))
print("nominal :", metrics.summarize_errors(fwd_err_nominal))
print("improved:", metrics.summarize_errors(fwd_err_improved))

729 visible test poses
nominal : {'n': 729, 'rms_angle': 0.032355279288777274, 'mean_angle': 0.02978185746651763, 'max_angle': 0.051654389277618944, 'rms_pos': 7.431406344784125, 'mean_pos': 6.790869277347467, 'max_pos': 14.886558949715186}
improved: {'n': 729, 'rms_angle': 0.00012744324891151074, 'mean_angle': 0.00011901692587762216, 'max_angle': 0.00020558367082204062, 'rms_pos': 0.06930846247709163, 'mean_pos': 0.06588374867048144, 'max_pos': 0.12076256045323701}


## Step 10: Evaluate `ImprovedInvKins`

For a target pose `F` (in the tracker frame), `theta =
ImprovedInvKins(F̂_X^-1 F)`, then actually move the robot there and compare
where it ends up — `mbee_placed.actual_placement` after the move, relative to
`F_trk` — with the target. Use targets you know are reachable (e.g.
ground-truth poses of test-grid joint vectors). The handout asks for the
time required for testing as well as for calibration, so keep track of it.
Note that every real move includes the joint repeatability noise
`sigma_rnd`; think about what floor that puts on this number.

In [12]:
rng = np.random.default_rng(0)
targets = [test_grid[i] for i in rng.choice(len(test_grid), size=min(40, len(test_grid)), replace=False)]

# Visit the targets in a greedy nearest-first order (by motion time): the same targets, less
# simulated testing time.
order, current, remaining = [], rob.theta_cmd.copy(), list(range(len(targets)))
while remaining:
    nxt = min(remaining, key=lambda i: rob.motion_time(current, targets[i]))
    order.append(nxt)
    remaining.remove(nxt)
    current = targets[nxt]
targets = [targets[i] for i in order]

t_test_start = clock.elapsed
F_X_imp = improved.base_frame()
inv_err_nominal, inv_err_improved = [], []
for theta_t in targets:
    F_target = F_trk.inv() * rob.actual_ee_pose(theta_t)   # a reachable target, in the tracker frame
    for kins, F_X, errs in ((rob, F_X_nom, inv_err_nominal), (improved, F_X_imp, inv_err_improved)):
        theta = kins.inv_kins(F_X.inv() * F_target, seed=theta_t)
        rob.move_joints(theta)
        actual = F_trk.inv() * mbee_placed.actual_placement   # ground truth, reporting only
        errs.append(metrics.frame_error(F_target, actual))

t_testing = clock.elapsed - t_test_start
print("nominal :", metrics.summarize_errors(inv_err_nominal))
print("improved:", metrics.summarize_errors(inv_err_improved))
print(f"simulated testing time: {t_testing:.1f}s for {2 * len(targets)} moves")

nominal : {'n': 40, 'rms_angle': 0.03466847820867715, 'mean_angle': 0.03230566920778643, 'max_angle': 0.05124924677767335, 'rms_pos': 7.691537041658795, 'mean_pos': 7.0303360606678975, 'max_pos': 13.088089912035926}
improved: {'n': 40, 'rms_angle': 0.0001764730791415555, 'mean_angle': 0.00016207779450536983, 'max_angle': 0.00039737291230949976, 'rms_pos': 0.07155839093219357, 'mean_pos': 0.06739369342081994, 'max_pos': 0.1133074084760801}
simulated testing time: 470.8s for 80 moves


## Achieved performance

Summarize: nominal vs improved, forward and inverse, against the handout's
targets (0.2 mm, 1e-3 rad), plus the simulated time spent calibrating and
testing and how the calibration time was spent (homing, number of
moves/samples). If a target is not met, say what you achieved and why —
that is an acceptable outcome per the handout. Put the same table in your
report, together with results for at least one other `SEED`.

In [13]:
def row(name, errs):
    s = metrics.summarize_errors(errs)
    ok = s["max_angle"] <= constants.TARGET_ANG_ERR_RAD and s["max_pos"] <= constants.TARGET_POS_ERR_MM
    return (f"{name:18s} {s['n']:5d} {s['rms_angle']:10.2e} {s['max_angle']:10.2e} "
            f"{s['rms_pos']:9.4f} {s['max_pos']:9.4f}   {'yes' if ok else 'no'}")

print(f"SEED {SEED}, plan {PLAN['method']}; targets {constants.TARGET_ANG_ERR_RAD} rad, "
      f"{constants.TARGET_POS_ERR_MM} mm (max)")
print(f"{'':18s} {'n':>5s} {'rms rad':>10s} {'max rad':>10s} {'rms mm':>9s} {'max mm':>9s}   met")
print(row("FwdK nominal", fwd_err_nominal))
print(row("FwdK improved", fwd_err_improved))
print(row("InvK nominal", inv_err_nominal))
print(row("InvK improved", inv_err_improved))
print(f"\ncalibration time {t_calibration:.1f}s = homing {t_home:.1f}s + baseline {t_baseline:.1f}s "
      f"+ data {data.t_collect:.1f}s ({len(poses)} readings, {n_distinct} distinct poses)")
print(f"testing time {t_testing:.1f}s")

SEED 1, plan single_axis; targets 0.001 rad, 0.2 mm (max)
                       n    rms rad    max rad    rms mm    max mm   met
FwdK nominal         729   3.24e-02   5.17e-02    7.4314   14.8866   no
FwdK improved        729   1.27e-04   2.06e-04    0.0693    0.1208   yes
InvK nominal          40   3.47e-02   5.12e-02    7.6915   13.0881   no
InvK improved         40   1.76e-04   3.97e-04    0.0716    0.1133   yes

calibration time 627.5s = homing 0.4s + baseline 83.8s + data 543.3s (1716 readings, 39 distinct poses)
testing time 470.8s


## Results over several seeds

The handout asks for results for more than one robot. `run_pipeline` repeats Steps 2 and 6-10
for a given `SEED` and plan without the diagnostic baseline: homing, plan, collection, fit,
forward evaluation over the test grid and inverse evaluation on the same 40 targets (improved
kinematics only). It resets the simulated clock, so run it last. Plan "single_axis" is the
design used above; "concurrent" (wrist joints 4, 5, 6 swept together with stages 1, 2, 3) is
the faster variant it is compared with.

In [14]:
def calibrate(seed, method="single_axis"):
    """Steps 2 and 6-8 for one seed, without the diagnostic baseline. Returns the robot,
    tracker, calibration body, fitted model, fit information and calibration time."""
    clock.reset()
    rob_s = simulation.create_robot("rob", F_rob_approx, seed=seed)
    trk_s = simulation.create_tracker("trk", F_tracker_approx, seed=seed)
    body_s = simulation.create_robot_calibration_body(rob_s, mbee_local_positions)
    simulation.add_marker_bodies(trk_s, [body_s])
    offsets = workflow.measure_home_offsets(rob_s, n_homings=5)
    plan = workflow.plan_calibration_poses(rob_s, trk_s, mbee_local_positions, DESIRED_ACCURACY, method=method)
    data_s = workflow.collect_calibration_data(rob_s, trk_s, body_s, plan, repeats=1)
    kins = kinematics.ImprovedKinematics(rob_s, workflow.estimate_base_frame(data_s, rob_s), joint_offsets=offsets)
    nominal_rms = kins._residual_rms(kins.initial_params(), data_s)   # nominal geometry at F_X_init
    info = kins.fit(data_s, F_tracker=F_tracker_approx)
    info["nominal_rms_rot_residual"], info["nominal_rms_pos_residual"] = nominal_rms
    return rob_s, trk_s, body_s, kins, info, clock.elapsed


def run_pipeline(seed, method="single_axis"):
    """calibrate, then the tracker-frame evaluation of Steps 9-10 (improved kinematics only)."""
    rob_s, trk_s, body_s, kins, info, t_cal = calibrate(seed, method)
    F_t = trk_s.nominal_pose()

    def visible(th):
        F = rob_s.F_rob * rob_s.fwd_kins(th)
        return trk_s.is_visible([F * a for a in mbee_local_positions])

    grid = [th for th in test_grid if visible(th)]
    fwd = [metrics.frame_error(F_t.inv() * rob_s.actual_ee_pose(th), kins.predict_in_tracker(th)) for th in grid]
    t0, inv, F_X_s = clock.elapsed, [], kins.base_frame()
    for th_t in targets:
        F_target = F_t.inv() * rob_s.actual_ee_pose(th_t)
        rob_s.move_joints(kins.inv_kins(F_X_s.inv() * F_target, seed=th_t))
        inv.append(metrics.frame_error(F_target, F_t.inv() * body_s.actual_placement))
    return dict(seed=seed, method=method, info=info, t_cal=t_cal, t_test=clock.elapsed - t0,
                fwd=metrics.summarize_errors(fwd), inv=metrics.summarize_errors(inv))


results = [run_pipeline(s, m) for m in ("single_axis", "concurrent") for s in (1, 2, 3, 4, 5)]
print(f"{'plan':12s} {'seed':>4s} {'start':>9s} {'t_cal s':>8s} {'t_test s':>8s} {'FwdK max rad':>12s} "
      f"{'FwdK max mm':>11s} {'InvK max rad':>12s} {'InvK max mm':>11s}")
for r in results:
    print(f"{r['method']:12s} {r['seed']:4d} {r['info']['init']:>9s} {r['t_cal']:8.1f} {r['t_test']:8.1f} "
          f"{r['fwd']['max_angle']:12.2e} {r['fwd']['max_pos']:11.4f} {r['inv']['max_angle']:12.2e} "
          f"{r['inv']['max_pos']:11.4f}")

plan         seed     start  t_cal s t_test s FwdK max rad FwdK max mm InvK max rad InvK max mm
single_axis     1 decoupled    528.8    435.0     1.78e-04      0.1302     3.94e-04      0.1158
single_axis     2 decoupled    528.8    435.0     2.34e-04      0.0986     3.78e-04      0.0943
single_axis     3 decoupled    528.8    435.0     2.33e-04      0.0954     4.36e-04      0.1088
single_axis     4 decoupled    528.8    435.0     2.03e-04      0.0822     5.50e-04      0.0917
single_axis     5 decoupled    528.8    435.0     1.35e-04      0.1305     5.25e-04      0.1275
concurrent      1   current    442.4    435.0     2.03e-04      0.1045     5.14e-04      0.1267
concurrent      2   current    442.4    435.0     2.29e-04      0.0985     3.73e-04      0.0991
concurrent      3   current    442.6    435.0     2.89e-04      0.1096     3.89e-04      0.1124
concurrent      4   current    442.6    435.0     1.74e-04      0.0873     3.54e-04      0.1042
concurrent      5   current    442.4    

## Extended evaluation

Ground truth is used here for reporting only. Three checks complement the template's evaluation:

1. **More seeds.** The single-axis pipeline of `run_pipeline` for seeds 6-25, with the fit
   statistics of all single-axis runs (evaluations, residuals at the decoupled and at the nominal start).
2. **Full joint ranges and the handout's base-frame definition.** For seeds 1-8, 600 random
   configurations over the full joint ranges: the forward error in the tracker frame (as in
   Step 9) and relative to the actual robot base, `(F*_ee(theta))^-1 ImprovedFwdKins(theta)` as
   written in the handout, both with the rotation about `d_1` chosen from the handout's bounds
   and with the fixed convention `phi_2 . x = 0` (`gauge_c = 0`); and 30 inverse-kinematics moves
   with targets given relative to the actual base.
3. The width of the admissible interval of the rotation about `d_1`.

The forward evaluation reads the simulator's ground truth only; the inverse evaluation moves the robot.

In [15]:
robust = [run_pipeline(s) for s in range(6, 26)]
single = [r for r in results if r["method"] == "single_axis"] + robust
worst = lambda runs, k, key: max(r[k][key] for r in runs)
print(f"single-axis plan, seeds 6-25 ({len(robust)} runs), tracker frame:")
print(f"  FwdK max {worst(robust, 'fwd', 'max_angle'):.2e} rad, {worst(robust, 'fwd', 'max_pos'):.4f} mm; "
      f"InvK max {worst(robust, 'inv', 'max_angle'):.2e} rad, {worst(robust, 'inv', 'max_pos'):.4f} mm")
print(f"  targets met in all runs: {all(r[k]['max_angle'] <= constants.TARGET_ANG_ERR_RAD and r[k]['max_pos'] <= constants.TARGET_POS_ERR_MM for r in robust for k in ('fwd', 'inv'))}")
rng_of = lambda key: (min(r["info"][key] for r in single), max(r["info"][key] for r in single))
print(f"single-axis plan, seeds 1-25: fit evaluations {rng_of('nfev')}")
print(f"  RMS residual at the decoupled start: {rng_of('init_rms_rot_residual')} rad, {rng_of('init_rms_pos_residual')} mm")
print(f"  RMS residual at the nominal start:   {rng_of('nominal_rms_rot_residual')} rad, {rng_of('nominal_rms_pos_residual')} mm")
print(f"  RMS residual after the fit:          {rng_of('rms_rot_residual')} rad, {rng_of('rms_pos_residual')} mm")
print(f"  calibration time {min(r['t_cal'] for r in single):.1f}-{max(r['t_cal'] for r in single):.1f} s")


def base_frame_evaluation(seed, n_fwd=600, n_inv=30):
    rob_s, trk_s, body_s, kins, info, _ = calibrate(seed)
    F_t = trk_s.nominal_pose()
    F_rob_true = rob_s.ground_truth_parameters()["F_rob_actual"]      # ground truth, reporting only
    rng = np.random.default_rng(100 + seed)
    c_bounds = kins.gauge_c
    trk_err, base_err, fixed_err = [], [], []
    for _ in range(n_fwd):
        th = rng.uniform(rob_s.theta_min, rob_s.theta_max)
        F_world = rob_s.actual_ee_pose(th)
        trk_err.append(metrics.frame_error(F_t.inv() * F_world, kins.predict_in_tracker(th)))
        base_err.append(metrics.frame_error(F_rob_true.inv() * F_world, kins.fwd_kins(th)))
        kins.gauge_c = 0.0                                              # fixed convention phi_2 . x = 0
        fixed_err.append(metrics.frame_error(F_rob_true.inv() * F_world, kins.fwd_kins(th)))
        kins.gauge_c = c_bounds
    inv_err = []
    for _ in range(n_inv):
        th_t = rng.uniform(0.9 * rob_s.theta_min, 0.9 * rob_s.theta_max)
        F_base = F_rob_true.inv() * rob_s.actual_ee_pose(th_t)         # target relative to the actual base
        rob_s.move_joints(kins.inv_kins(F_base, seed=th_t))
        inv_err.append(metrics.frame_error(F_base, F_rob_true.inv() * body_s.actual_placement))
    lo_hi = info["gauge_interval"]
    return dict(seed=seed, trk=metrics.summarize_errors(trk_err), base=metrics.summarize_errors(base_err),
                fixed=metrics.summarize_errors(fixed_err), inv_base=metrics.summarize_errors(inv_err),
                width=(lo_hi[1] - lo_hi[0]) if lo_hi else float("nan"))


base_results = [base_frame_evaluation(s) for s in range(1, 9)]
print(f"\n600 random full-range configurations per seed; maxima (rad, mm)")
print(f"{'seed':>4s} {'tracker':>17s} {'base, bounds':>17s} {'base, fixed':>17s} {'InvK base':>17s} {'width c':>8s}")
for b in base_results:
    cols = [f"{b[k]['max_angle']:.2e} {b[k]['max_pos']:7.3f}" for k in ("trk", "base", "fixed", "inv_base")]
    print(f"{b['seed']:4d} " + " ".join(f"{c:>17s}" for c in cols) + f" {b['width']:8.4f}")

single-axis plan, seeds 6-25 (20 runs), tracker frame:
  FwdK max 2.93e-04 rad, 0.1460 mm; InvK max 5.25e-04 rad, 0.1420 mm
  targets met in all runs: True
single-axis plan, seeds 1-25: fit evaluations (9, 19)
  RMS residual at the decoupled start: (0.00034083894306580215, 0.00046499251052939026) rad, (0.01666055305056429, 0.026195418910091986) mm
  RMS residual at the nominal start:   (0.013923877324904205, 0.02337071279841307) rad, (2.464916425581249, 4.852910343602129) mm
  RMS residual after the fit:          (0.00032986877280018396, 0.0004590904976729587) rad, (0.014711333931078952, 0.022699647684650097) mm
  calibration time 528.8-528.8 s

600 random full-range configurations per seed; maxima (rad, mm)
seed           tracker      base, bounds       base, fixed         InvK base  width c
   1  2.25e-04   0.143  3.32e-04   0.110  1.56e-02  11.642  4.46e-04   0.089   0.0053
   2  2.10e-04   0.127  8.66e-04   0.556  3.38e-03   2.451  1.00e-03   0.485   0.0061
   3  2.39e-04   0.089  